<a href="https://colab.research.google.com/github/Subho7439/paper_model/blob/main/paper_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Adaptive Confidence-Guided Explainable Hybrid CNN-ViT Framework
for Brain Tumor Classification Using MRI Images.

COLAB-READY VERSION: paste this whole file into a Colab cell (or a few
cells split at the "# ---" markers) and run top to bottom. It:
  1. Installs deps
  2. Downloads the Kaggle dataset (sartajbhuvaji/brain-tumor-classification-mri)
  3. Builds + trains + evaluates the hybrid CNN-ViT model from the paper
  4. Produces Grad-CAM and SHAP explanations on a few test images

Architecture maps to the paper as:
  CNN branch                              -> Section 4.4, Eq. (1)
  ViT branch                              -> Section 4.5, Eq. (2)
  Adaptive confidence-guided fusion       -> Section 4.6, Eq. (3)-(6)
  Classification head                     -> Section 4.7, Eq. (7)-(8)
  Explainability (Grad-CAM + SHAP)        -> Section 4.8
"""

# --------------------------------------------------------------------------
# 0. Install dependencies (Colab already has tensorflow; the rest are extra)
# --------------------------------------------------------------------------
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                 "kagglehub", "shap", "opencv-python-headless", "seaborn"])

import os
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# --------------------------------------------------------------------------
# 1. Download dataset and locate Training/Testing folders
# --------------------------------------------------------------------------
import kagglehub

DATASET_SLUG = "sartajbhuvaji/brain-tumor-classification-mri"
download_root = kagglehub.dataset_download(DATASET_SLUG)
print("Downloaded to:", download_root)


def find_data_dir(root):
    """Locate the folder that directly contains Training/ and Testing/."""
    for dirpath, dirnames, _ in os.walk(root):
        if "Training" in dirnames and "Testing" in dirnames:
            return dirpath
    raise FileNotFoundError(
        f"Could not find Training/ and Testing/ folders under {root}. "
        f"Contents: {os.listdir(root)}"
    )


DATA_DIR = find_data_dir(download_root)
print("Using DATA_DIR:", DATA_DIR)
print("Training classes found:", sorted(os.listdir(os.path.join(DATA_DIR, "Training"))))

# --------------------------------------------------------------------------
# 2. Config
# --------------------------------------------------------------------------
IMG_SIZE = 224
# This is the ACTUAL class-folder naming used by sartajbhuvaji's dataset.
CLASS_NAMES = ["glioma_tumor", "meningioma_tumor", "no_tumor", "pituitary_tumor"]
NUM_CLASSES = len(CLASS_NAMES)

PATCH_SIZE = 16
NUM_PATCHES = (IMG_SIZE // PATCH_SIZE) ** 2      # 196
PROJ_DIM = 256
TRANSFORMER_LAYERS = 6
NUM_HEADS = 8
MLP_DIM = 512
DROPOUT = 0.1

EPOCHS = 40
BATCH_SIZE = 32
LR = 1e-4

AUTOTUNE = tf.data.AUTOTUNE

# --------------------------------------------------------------------------
# 3. Data pipeline (Section 4.3: resize, normalize, augment)
# --------------------------------------------------------------------------
def build_datasets(data_dir, batch_size=BATCH_SIZE, val_split=0.15, seed=42):
    train_dir = os.path.join(data_dir, "Training")
    test_dir = os.path.join(data_dir, "Testing")

    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, validation_split=val_split, subset="training", seed=seed,
        label_mode="categorical", class_names=CLASS_NAMES,
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=batch_size,
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, validation_split=val_split, subset="validation", seed=seed,
        label_mode="categorical", class_names=CLASS_NAMES,
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=batch_size,
    )
    test_ds = tf.keras.utils.image_dataset_from_directory(
        test_dir, label_mode="categorical", class_names=CLASS_NAMES,
        image_size=(IMG_SIZE, IMG_SIZE), batch_size=batch_size, shuffle=False,
    )

    augment = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.06),
        layers.RandomZoom(0.10),
        layers.RandomTranslation(0.08, 0.08),
        layers.RandomContrast(0.08),
    ], name="augmentation")

    def prep(x, y, training):
        x = tf.cast(x, tf.float32)                 # keep raw 0-255 scale
        if training:
            x = augment(x, training=True)
        # each branch does its own internal preprocessing (see model below)
        return x, {"final_output": y, "cnn_aux": y, "vit_aux": y}

    train_ds = train_ds.map(lambda x, y: prep(x, y, True), num_parallel_calls=AUTOTUNE)
    val_ds = val_ds.map(lambda x, y: prep(x, y, False), num_parallel_calls=AUTOTUNE)
    test_ds_labeled = test_ds.map(lambda x, y: prep(x, y, False), num_parallel_calls=AUTOTUNE)

    train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
    val_ds = val_ds.cache().prefetch(AUTOTUNE)
    test_ds_labeled = test_ds_labeled.prefetch(AUTOTUNE)

    return train_ds, val_ds, test_ds_labeled, test_ds


# --------------------------------------------------------------------------
# 4. ViT branch -> Eq. (2): F_vit = ViT(I)
# --------------------------------------------------------------------------
class Patches(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        batch_size = tf.shape(images)[0]
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        patch_dims = patches.shape[-1]
        return tf.reshape(patches, [batch_size, -1, patch_dims])

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"patch_size": self.patch_size})
        return cfg


class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection = layers.Dense(projection_dim)
        self.position_embedding = layers.Embedding(input_dim=num_patches, output_dim=projection_dim)

    def call(self, patch):
        positions = tf.range(start=0, limit=self.num_patches, delta=1)
        return self.projection(patch) + self.position_embedding(positions)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"num_patches": self.num_patches, "projection_dim": self.projection.units})
        return cfg


def transformer_encoder_block(x, projection_dim, num_heads, mlp_dim, dropout=DROPOUT):
    x1 = layers.LayerNormalization(epsilon=1e-6)(x)
    attn = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=projection_dim // num_heads, dropout=dropout
    )(x1, x1)
    x2 = layers.Add()([attn, x])
    x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
    mlp = layers.Dense(mlp_dim, activation=tf.nn.gelu)(x3)
    mlp = layers.Dropout(dropout)(mlp)
    mlp = layers.Dense(projection_dim)(mlp)
    mlp = layers.Dropout(dropout)(mlp)
    return layers.Add()([mlp, x2])


def build_vit_branch(raw_input):
    x = layers.Rescaling(1.0 / 255.0, name="vit_rescale")(raw_input)
    patches = Patches(PATCH_SIZE, name="vit_patches")(x)
    encoded = PatchEncoder(NUM_PATCHES, PROJ_DIM, name="vit_patch_encoder")(patches)
    h = encoded
    for _ in range(TRANSFORMER_LAYERS):
        h = transformer_encoder_block(h, PROJ_DIM, NUM_HEADS, MLP_DIM)
    h = layers.LayerNormalization(epsilon=1e-6)(h)
    f_vit = layers.GlobalAveragePooling1D(name="vit_gap")(h)   # F_vit
    return f_vit


# --------------------------------------------------------------------------
# 5. CNN branch -> Eq. (1): F_cnn = CNN(I)
# --------------------------------------------------------------------------
def build_cnn_branch(raw_input, fine_tune_last_n=30):
    x = layers.Lambda(
        tf.keras.applications.efficientnet.preprocess_input, name="cnn_preprocess"
    )(raw_input)
    base = tf.keras.applications.EfficientNetB0(
        include_top=False, weights="imagenet", pooling="avg",
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    base._name = "cnn_backbone"
    for layer in base.layers[:-fine_tune_last_n]:
        layer.trainable = False
    f_cnn_raw = base(x)
    f_cnn = layers.Dense(PROJ_DIM, activation="relu", name="cnn_projection")(f_cnn_raw)
    return f_cnn, base


# --------------------------------------------------------------------------
# 6. Adaptive Confidence-Guided Fusion -> Eq. (3)-(6)
# --------------------------------------------------------------------------
class ConfidenceGuidedFusion(layers.Layer):
    """
    C_i    = softmax confidence of branch i (max class probability)  (Eq. 3)
    alpha  = C_cnn / (C_cnn + C_vit)                                  (Eq. 4)
    beta   = C_vit / (C_cnn + C_vit)                                  (Eq. 4)
    F_fuse = alpha * F_cnn + beta * F_vit                             (Eq. 5)
    alpha + beta = 1                                                  (Eq. 6)
    """
    def call(self, inputs):
        f_cnn, f_vit, logits_cnn, logits_vit = inputs
        p_cnn = tf.nn.softmax(logits_cnn, axis=-1)
        p_vit = tf.nn.softmax(logits_vit, axis=-1)
        c_cnn = tf.reduce_max(p_cnn, axis=-1, keepdims=True)   # Eq. 3
        c_vit = tf.reduce_max(p_vit, axis=-1, keepdims=True)   # Eq. 3
        denom = c_cnn + c_vit + 1e-8
        alpha = c_cnn / denom                                    # Eq. 4
        beta = c_vit / denom                                     # Eq. 4
        fused = alpha * f_cnn + beta * f_vit                     # Eq. 5
        return fused, alpha, beta


# --------------------------------------------------------------------------
# 7. Full model assembly
# --------------------------------------------------------------------------
def build_acghv_model():
    inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="mri_input")

    f_cnn, cnn_backbone = build_cnn_branch(inputs)
    f_vit = build_vit_branch(inputs)

    logits_cnn = layers.Dense(NUM_CLASSES, name="cnn_logits")(f_cnn)
    logits_vit = layers.Dense(NUM_CLASSES, name="vit_logits")(f_vit)

    fused, alpha, beta = ConfidenceGuidedFusion(name="confidence_fusion")(
        [f_cnn, f_vit, logits_cnn, logits_vit]
    )

    x = layers.Dense(256, activation="relu")(fused)
    x = layers.Dropout(0.3)(x)
    final_logits = layers.Dense(NUM_CLASSES, name="final_logits")(x)

    final_output = layers.Activation("softmax", name="final_output")(final_logits)   # Eq. 7
    cnn_aux = layers.Activation("softmax", name="cnn_aux")(logits_cnn)
    vit_aux = layers.Activation("softmax", name="vit_aux")(logits_vit)

    model = Model(inputs=inputs, outputs=[final_output, cnn_aux, vit_aux], name="ACGHV")
    fusion_inspector = Model(inputs=inputs, outputs=[final_output, alpha, beta], name="ACGHV_inspect")

    return model, fusion_inspector, cnn_backbone


def compile_model(model, lr=LR):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss={
            "final_output": "categorical_crossentropy",   # Eq. 8, primary loss
            "cnn_aux": "categorical_crossentropy",
            "vit_aux": "categorical_crossentropy",
        },
        loss_weights={"final_output": 1.0, "cnn_aux": 0.3, "vit_aux": 0.3},
        metrics={"final_output": ["accuracy"]},
    )
    return model


# --------------------------------------------------------------------------
# 8. Training
# --------------------------------------------------------------------------
def train(model, train_ds, val_ds, epochs=EPOCHS, ckpt_path="best_acghv.keras"):
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_final_output_accuracy", patience=8,
            restore_best_weights=True, mode="max"
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_final_output_loss", factor=0.5, patience=4, min_lr=1e-7, mode="min"
        ),
        tf.keras.callbacks.ModelCheckpoint(
            ckpt_path, monitor="val_final_output_accuracy",
            save_best_only=True, mode="max"
        ),
    ]
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks)
    return history


# --------------------------------------------------------------------------
# 9. Evaluation (Section 5.4 / 6.1-6.3)
# --------------------------------------------------------------------------
def evaluate(model, test_ds_labeled):
    y_true, y_pred_probs = [], []
    for batch_x, batch_y in test_ds_labeled:
        preds = model.predict(batch_x, verbose=0)[0]   # final_output branch
        y_pred_probs.append(preds)
        y_true.append(batch_y["final_output"].numpy())
    y_true = np.concatenate(y_true)
    y_pred_probs = np.concatenate(y_pred_probs)
    y_true_labels = np.argmax(y_true, axis=1)
    y_pred_labels = np.argmax(y_pred_probs, axis=1)

    print(classification_report(y_true_labels, y_pred_labels, target_names=CLASS_NAMES, digits=4))

    cm = confusion_matrix(y_true_labels, y_pred_labels)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion Matrix")
    plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=150)
    plt.show()

    specificity = {}
    for i, cls in enumerate(CLASS_NAMES):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        specificity[cls] = tn / (tn + fp + 1e-8)
    print("Specificity per class:", specificity)

    y_true_bin = label_binarize(y_true_labels, classes=list(range(NUM_CLASSES)))
    plt.figure(figsize=(6, 5))
    aucs = []
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        plt.plot(fpr, tpr, label=f"{cls} (AUC={roc_auc:.3f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curves (One-vs-Rest)"); plt.legend()
    plt.tight_layout(); plt.savefig("roc_curves.png", dpi=150)
    plt.show()
    print(f"Macro AUC-ROC: {np.mean(aucs):.4f}")

    overall_acc = (y_true_labels == y_pred_labels).mean()
    print(f"Overall test accuracy: {overall_acc*100:.2f}%")
    return overall_acc


# --------------------------------------------------------------------------
# 10. Explainability -> Section 4.8: Grad-CAM (CNN branch) + SHAP
# --------------------------------------------------------------------------
def grad_cam(model, cnn_backbone, image, class_index=None, last_conv_layer_name="top_conv"):
    grad_model = Model(
        inputs=model.inputs,
        outputs=[cnn_backbone.get_layer(last_conv_layer_name).output, model.outputs[0]],
    )
    img_batch = tf.expand_dims(image, axis=0)
    with tf.GradientTape() as tape:
        conv_output, predictions = grad_model(img_batch)
        if class_index is None:
            class_index = tf.argmax(predictions[0])
        class_channel = predictions[:, class_index]

    grads = tape.gradient(class_channel, conv_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_output = conv_output[0]
    heatmap = conv_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def plot_grad_cam(image, heatmap, out_path="gradcam_overlay.png", alpha=0.4):
    import cv2
    heatmap_resized = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap_resized)
    heatmap_color = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    img_uint8 = np.uint8(np.clip(image, 0, 255))
    overlay = cv2.addWeighted(cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR), 1 - alpha, heatmap_color, alpha, 0)
    cv2.imwrite(out_path, overlay)
    return overlay


def shap_explain(model, background_images, sample_images, out_path="shap_summary.png"):
    import shap
    explainer = shap.GradientExplainer((model.inputs[0], model.outputs[0]), background_images)
    shap_values = explainer.shap_values(sample_images)
    shap.image_plot(shap_values, sample_images, show=False)
    plt.savefig(out_path, dpi=150)
    plt.show()
    return shap_values


# --------------------------------------------------------------------------
# 11. Run everything
# --------------------------------------------------------------------------
train_ds, val_ds, test_ds_labeled, test_ds_raw = build_datasets(DATA_DIR)

model, fusion_inspector, cnn_backbone = build_acghv_model()
model = compile_model(model)
model.summary()

history = train(model, train_ds, val_ds, epochs=EPOCHS)

test_accuracy = evaluate(model, test_ds_labeled)

# Grad-CAM + SHAP on a handful of test images
sample_batch_x, _ = next(iter(test_ds_labeled))
grad_cam_map = grad_cam(model, cnn_backbone, sample_batch_x[0])
plot_grad_cam(sample_batch_x[0].numpy(), grad_cam_map)

try:
    background = sample_batch_x[:8].numpy()
    samples = sample_batch_x[8:11].numpy()
    shap_explain(model, background, samples)
except Exception as e:
    print("SHAP explanation skipped (optional, can be slow):", e)

print(f"\nDone. Final test accuracy: {test_accuracy*100:.2f}%")
print("Saved: confusion_matrix.png, roc_curves.png, gradcam_overlay.png, shap_summary.png, best_acghv.keras")

TF version: 2.20.0
GPU: []
Using Colab cache for faster access to the 'brain-tumor-classification-mri' dataset.
Downloaded to: /kaggle/input/brain-tumor-classification-mri
Using DATA_DIR: /kaggle/input/brain-tumor-classification-mri
Training classes found: ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
Found 2870 files belonging to 4 classes.
Using 2440 files for training.
Found 2870 files belonging to 4 classes.
Using 430 files for validation.
Found 394 files belonging to 4 classes.


Model: "ACGHV"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mri_input           │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vit_rescale         │ (None, 224, 224,  │          0 │ mri_input[0][0]   │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vit_patches         │ (None, None, 768) │          0 │ vit_rescale[0][0] │
│ (Patches)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vit_patch_encoder   │ (None, 196, 256)  │    247,040 │ vit_patches[0][0] │
│ (PatchEncoder)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ vit_patch_encode… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 196, 256)  │    263,168 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_24 (Add)        │ (None, 196, 256)  │          0 │ multi_head_atten… │
│                     │                   │            │ vit_patch_encode… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ add_24[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_29 (Dense)    │ (None, 196, 512)  │    131,584 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_39          │ (None, 196, 512)  │          0 │ dense_29[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_30 (Dense)    │ (None, 196, 256)  │    131,328 │ dropout_39[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_40          │ (None, 196, 256)  │          0 │ dense_30[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_25 (Add)        │ (None, 196, 256)  │          0 │ dropout_40[0][0], │
│                     │                   │            │ add_24[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ add_25[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 196, 256)  │    263,168 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_26 (Add)        │ (None, 196, 256)  │          0 │ multi_head_atten… │
│                     │                   │            │ add_25[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ add_26[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 7,856,559 (29.97 MB)

 Trainable params: 5,300,844 (20.22 MB)

 Non-trainable params: 2,555,715 (9.75 MB)

Epoch 1/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 894s 10s/step - cnn_aux_loss: 0.9636 - final_output_accuracy: 0.5750 - final_output_loss: 1.0305 - loss: 1.6817 - vit_aux_loss: 1.2049 - val_cnn_aux_loss: 0.7246 - val_final_output_accuracy: 0.7512 - val_final_output_loss: 0.6369 - val_loss: 1.1807 - val_vit_aux_loss: 1.1116 - learning_rate: 1.0000e-04
Epoch 2/40
77/77 ━━━━━━━━━━━━━━━━━━━━ 801s 10s/step - cnn_aux_loss: 0.5276 - final_output_accuracy: 0.7885 - final_output_loss: 0.5640 - loss: 1.0567 - vit_aux_loss: 1.1144 - val_cnn_aux_loss: 0.5459 - val_final_output_accuracy: 0.7419 - val_final_output_loss: 0.6471 - val_loss: 1.1567 - val_vit_aux_loss: 1.1534 - learning_rate: 1.0000e-04
Epoch 3/40
 9/77 ━━━━━━━━━━━━━━━━━━━━ 10:43 9s/step - cnn_aux_loss: 0.4281 - final_output_accuracy: 0.8096 - final_output_loss: 0.4702 - loss: 0.9214 - vit_aux_loss: 1.0758 